In [16]:
import arcpy
import os
import pandas as pd
import requests
import zipfile
import shutil

arcpy.management.ClearWorkspaceCache()
arcpy.env.overwriteOutput=True

In [17]:
root_dir = os.getcwd()
gdbDir = os.path.join(root_dir, "NOAAChloroplethMap")
gdbName = "NOAA.gdb"
projectGDB = os.path.join(gdbDir, gdbName)
projectPath = os.path.join(gdbDir, "NOAAFloodChloropleth.aprx")


os.makedirs(gdbDir, exist_ok=True)
if not os.path.exists(projectGDB):
    arcpy.management.CreateFileGDB(gdbDir, gdbName)


countyBoundariesURL = "https://services.arcgis.com/vPD5PVLI6sfkZ5E4/arcgis/rest/services/Iowa_County_Boundaries_Census/FeatureServer/0"
watershedBoundariesURL = "https://iowageodata2.s3.us-east-2.amazonaws.com/inlandWaters/watersheds/WBD_HU_08_10_12_IA.zip"

boundariesGDBpath = os.path.join(projectGDB, "Iowa_County_Boundaries_Census")
if not arcpy.Exists(boundariesGDBpath):
    arcpy.management.CopyFeatures(countyBoundariesURL, boundariesGDBpath)

watershedGDBpath = os.path.join(projectGDB, "WBD_HU_08_IA")
if not arcpy.Exists(watershedGDBpath):
    zip_filename = watershedBoundariesURL.split("/")[-1]
    temp_zip_path = os.path.join(gdbDir, zip_filename)
    
    response = requests.get(watershedBoundariesURL, timeout=30)
    if response.status_code == 200:
        with open(temp_zip_path, "wb") as f:
            f.write(response.content)
        with zipfile.ZipFile(temp_zip_path, 'r') as zip_ref:
            zip_ref.extractall(gdbDir)
        
        extracted_shp = os.path.join(gdbDir, "WBD_HU_08_IA.shp")
        arcpy.management.CopyFeatures(extracted_shp, watershedGDBpath)
        os.remove(temp_zip_path)

spatial_ref = arcpy.SpatialReference(26915) # NAD 1983 UTM Zone 15N (Standard for Iowa)

projected_counties = os.path.join(projectGDB, "Counties_Projected")
projected_watersheds = os.path.join(projectGDB, "Watersheds_Projected")

if not arcpy.Exists(projected_counties):
    arcpy.management.Project(boundariesGDBpath, projected_counties, spatial_ref)

if not arcpy.Exists(projected_watersheds):
    arcpy.management.Project(watershedGDBpath, projected_watersheds, spatial_ref)
else:
    print('exists')

username = os.getlogin()
blank_template = r'C:\Users\{username}\AppData\Local\Programs\ArcGIS\Pro\Resources\ArcToolBox\Services\routingservices\data\Blank.aprx'
if not os.path.exists(projectPath):
    shutil.copy(blank_template, projectPath)

aprx = arcpy.mp.ArcGISProject(projectPath)


exists


In [18]:
cMap = aprx.listMaps()[0]

In [19]:
floodCountsPath = os.path.join(gdbDir, "countPerCountyClean.csv")
if not os.path.exists(floodCountsPath):
    floodCounts = pd.read_csv('countPerCounty.csv')
    floodCounts['CZ_NAME'] = floodCounts["CZ_NAME"].str.replace(' COUNTY', '').str.strip().str.upper()
    floodCounts.to_csv(floodCountsPath)
else:
    print (f"{floodCountsPath} is present")

c:\Users\cfuchtman\Desktop\IA_Flood_Compendium\NOAAChloroplethMap\countPerCountyClean.csv is present


In [20]:
finalJoinedCounty = os.path.join(projectGDB, "Counties_with_NOAA_Data")
if not arcpy.Exists(finalJoinedCounty):
    print("Standardizing county join keys and executing database join...")

    arcpy.management.MakeFeatureLayer(projected_counties, "temp_layer")
    
  
    arcpy.management.CalculateField(
        in_table="temp_layer",
        field="NAME",
        expression="!NAME!.upper()",
        expression_type="PYTHON3"
    )
    
    # Join the local CSV straight into the temporary feature class layer
    arcpy.management.AddJoin(
        in_layer_or_view="temp_layer",
        in_field="NAME",
        join_table=floodCountsPath,
        join_field="CZ_NAME"
    )
    
    arcpy.management.CopyFeatures("temp_layer", finalJoinedCounty)
    print("Data join finalized and stored successfully in NOAA.gdb.")
    

    arcpy.management.Delete("temp_layer")
else:
    print("already done")



already done


In [21]:
existingLayers = [layer.name for layer in cMap.listLayers()]

if 'Counties_with_NOAA_Data' not in existingLayers:
    cMap.addDataFromPath(finalJoinedCounty)
else:
    print('already don')

already don


In [22]:
layer_to_style = cMap.listLayers('Counties_with_NOAA_Data')[0]
symbology = layer_to_style.symbology

if hasattr(symbology, 'renderer'):
    symbology.updateRenderer('GraduatedColorsRenderer')
    symbology.renderer.classificationField = 'countPerCountyClean_csv_FLOOD_COUNT'
    symbology.renderer.classificationMethod = 'NaturalBreaks'
    symbology.renderer.colorRamp = aprx.listColorRamps('Reds')[0]
    layer_to_style.symbology = symbology

aprx.save()
